# Model Explainability Analysis

This notebook demonstrates various explainability techniques including SHAP, Grad-CAM, and LIME.

## Import Libraries

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import json

# Explainability libraries
try:
    import shap
    import lime
    import lime.lime_image
    print("Explainability libraries imported successfully")
except ImportError:
    print("Install shap and lime for explainability features")

## Load Model

In [ ]:
# Load pre-trained model
model_path = "model_weights/cnn_ulcer_model.pth"

if Path(model_path).exists():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    # Load model (placeholder)
    print(f"Model would be loaded from {model_path}")
else:
    print(f"Model not found at {model_path}")

## SHAP Values Explanation

In [ ]:
# Load SHAP values if available
shap_file = "shap_explanations/shap_values.json"

try:
    with open(shap_file, 'r') as f:
        shap_data = json.load(f)
    
    print(f"Loaded SHAP data for {len(shap_data)} images")
    
    # Display first result
    first_result = shap_data[0]
    print(f"Image: {first_result['image_path']}")
    print(f"Prediction: {first_result['prediction']:.4f}")
    print(f"Feature Importance: {first_result['feature_importance']}")
except FileNotFoundError:
    print("SHAP values file not found")

## Grad-CAM Visualization

In [ ]:
class GradCAM:
    def __init__(self, model, layer_name):
        self.model = model
        self.layer_name = layer_name
        self.gradients = None
        self.activations = None
    
    def forward_hook(self, module, input, output):
        self.activations = output.detach()
    
    def backward_hook(self, module, grad_input, grad_output):
        self.gradients = grad_output[0].detach()
    
    def generate_cam(self, input_image):
        # Generate class activation map
        weights = self.gradients.mean(dim=(2, 3), keepdim=True)
        cam = (weights * self.activations).sum(dim=1, keepdim=True)
        cam = torch.relu(cam)
        return cam.squeeze().cpu().numpy()

print("Grad-CAM class defined")

## Feature Importance Analysis

In [ ]:
# Analyze feature importance from SHAP values
try:
    with open("shap_explanations/summary.json", 'r') as f:
        summary = json.load(f)
    
    importance = summary.get('feature_importance_summary', {})
    
    # Plot importance
    fig, ax = plt.subplots(figsize=(10, 6))
    features = list(importance.keys())
    values = list(importance.values())
    
    ax.barh(features, values, color='steelblue')
    ax.set_xlabel('Importance Score')
    ax.set_title('Feature Importance from SHAP')
    plt.tight_layout()
    plt.show()
    
    print(f"Total images processed: {summary.get('processed_images')}")
    print(f"Average prediction: {summary.get('average_prediction'):.4f}")
except FileNotFoundError:
    print("Summary file not found")

## Model Confidence Analysis

In [ ]:
# Analyze model confidence distribution
try:
    with open("shap_explanations/shap_values.json", 'r') as f:
        shap_data = json.load(f)
    
    predictions = [r['prediction'] for r in shap_data]
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Histogram
    axes[0].hist(predictions, bins=20, color='steelblue', edgecolor='black')
    axes[0].set_xlabel('Prediction Confidence')
    axes[0].set_ylabel('Frequency')
    axes[0].set_title('Distribution of Model Predictions')
    
    # Statistics
    stats_text = f"""Model Prediction Statistics:
Mean: {np.mean(predictions):.4f}
Std: {np.std(predictions):.4f}
Min: {np.min(predictions):.4f}
Max: {np.max(predictions):.4f}
Median: {np.median(predictions):.4f}"""
    
    axes[1].text(0.1, 0.5, stats_text, fontsize=12, verticalalignment='center',
                 family='monospace', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
    axes[1].axis('off')
    
    plt.tight_layout()
    plt.show()
except:
    print("Could not analyze confidence")

## Conclusions

In [ ]:
print("""
Explainability Analysis Summary:
================================

1. SHAP Values: Provide global feature importance
   - Used for understanding which features drive predictions
   - Can be computed for both image and clinical features

2. Grad-CAM: Visual explanations for CNN predictions
   - Shows which regions of the image are important
   - Useful for validating model focus areas

3. LIME: Local interpretable explanations
   - Explains individual predictions
   - Helps identify prediction artifacts

4. Feature Importance: Aggregated analysis
   - Identifies most impactful features
   - Guides feature engineering and data collection
""")